# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GraphQL enrichment (`pr_title`, `pr_body`, `repo_star_count`, `is_resolved`) → unified code enrichment (compare patches, base/snapshot zipballs, patched content, dependency resolution).

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [1]:
import asyncio
import logging

import json
import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    github_graphql,
)
from ai_code_reviewer.dataset import config as dataset_config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [2]:
gh_archive_semaphore = asyncio.Semaphore(dataset_config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(dataset_config.GITHUB_API_CONCURRENCY)

In [3]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GH_ARCHIVE_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        dataset_config.RANGE_START,
        dataset_config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_RAW_PATH)

Fetching hourly data: 100%|██████████| 1/1 [00:04<00:00,  4.72s/it]
INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/dataset_raw.json.gz')

In [4]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset, 5#dataset_config.SNAPSHOT_COMMITS_TO_KEEP
)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FILTERED_PATH)

INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/filtered.json.gz')

In [5]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_graphql.enrich_dataset_with_graphql_info(
        dataset, session, gh_semaphore
    )

GraphQL enrichment: 100%|██████████| 5/5 [00:00<00:00,  8.79it/s]


In [6]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_api.enrich_dataset_with_code(
        dataset, session, gh_semaphore
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FINAL_PATH)

Enriching repos: 100%|██████████| 3/3 [01:59<00:00, 39.91s/it]
INFO:ai_code_reviewer.dataset.checkpoints:Wrote checkpoint -> /Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/final.json.gz


PosixPath('/Users/nikita/University/DataMining/Project/ai-code-reviewer/data/checkpoints/final.json.gz')

In [8]:
# Validate dependency content source:
#   - If a dependency file was itself changed in the same snapshot commit,
#     its content in `dependencies` is the annotated patched view (+/- markers).
#   - If a dependency file was NOT changed, its content is raw HEAD text from
#     the snapshot zipball.
patched_src_deps: list[tuple[str, str, str, str, str]] = []
zipball_src_deps: list[tuple[str, str, str, str, str]] = []

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        for commit_sha, path_map in pr_entry["commits"].items():
            for path, file_entry in path_map.items():
                deps = file_entry.get("dependencies") or {}
                for dep_path, dep_content in deps.items():
                    record = (repo_name, pr_number, commit_sha[:7], path, dep_path)
                    if dep_path in path_map:
                        patched_src_deps.append(record)
                    else:
                        zipball_src_deps.append(record)

print(f"Dependencies sourced from patched_content (dep also changed in snapshot): {len(patched_src_deps)}")
for repo, pr, sha, src, dep in patched_src_deps[:5]:
    print(f"  [{repo} PR#{pr} @{sha}]  {src}  ->  {dep}")

print()
print(f"Dependencies sourced from snapshot zipball (unchanged dep files):          {len(zipball_src_deps)}")
for repo, pr, sha, src, dep in zipball_src_deps[:5]:
    print(f"  [{repo} PR#{pr} @{sha}]  {src}  ->  {dep}")

Dependencies sourced from patched_content (dep also changed in snapshot): 0

Dependencies sourced from snapshot zipball (unchanged dep files):          12
  [tomwojcik/starlette-context PR#59 @37cafcf]  starlette_context/middleware/mixin.py  ->  starlette_context/errors.py
  [tomwojcik/starlette-context PR#59 @37cafcf]  tests/test_plugins/test_error_responses.py  ->  starlette_context/__init__.py
  [tomwojcik/starlette-context PR#59 @37cafcf]  tests/test_plugins/test_error_responses.py  ->  starlette_context/header_keys.py
  [tomwojcik/starlette-context PR#59 @37cafcf]  tests/test_plugins/test_error_responses.py  ->  starlette_context/middleware/__init__.py
  [tomwojcik/starlette-context PR#59 @37cafcf]  tests/test_plugins/test_error_responses.py  ->  starlette_context/plugins/__init__.py


In [9]:
# Compute dataset statistics
from ai_code_reviewer.dataset import config as dataset_config

num_prs = 0
num_snapshot_commits = 0
num_files_with_comments = 0
num_files_without_comments = 0
num_resolved_comments = 0
num_unresolved_comments = 0
balance_violations = 0  # snapshots where no-comment files exceed commented files
num_files_with_deps = 0
total_dep_files = 0
num_commits_with_metadata = 0
total_metadata_files = 0

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        num_prs += 1
        for commit_sha, path_map in pr_entry["commits"].items():
            num_snapshot_commits += 1
            snap_with = 0
            snap_without = 0
            for path, file_entry in path_map.items():
                if path == dataset_config.METADATA_FILES_COMMIT_KEY:
                    num_commits_with_metadata += 1
                    total_metadata_files += len(file_entry)
                    continue
                comments = file_entry.get("comments", [])
                for comment in comments:
                    if comment.get("is_resolved", False):
                        num_resolved_comments += 1
                    else:
                        num_unresolved_comments += 1
                if comments:
                    snap_with += 1
                else:
                    snap_without += 1
                deps = file_entry.get("dependencies", {})
                if deps:
                    num_files_with_deps += 1
                    total_dep_files += len(deps)
            num_files_with_comments += snap_with
            num_files_without_comments += snap_without
            if snap_without > snap_with:
                balance_violations += 1

num_files = num_files_with_comments + num_files_without_comments
num_comments = num_resolved_comments + num_unresolved_comments

print(f"Number of PRs:                  {num_prs}")
print(f"Number of snapshot commits:     {num_snapshot_commits}")
print(f"Number of files (total):        {num_files}")
print(f"  - with comments:              {num_files_with_comments}")
print(f"  - without comments:           {num_files_without_comments}")
print(f"Number of comments (total):     {num_comments}")
print(f"  - resolved:                   {num_resolved_comments}")
print(f"  - unresolved:                 {num_unresolved_comments}")
print(
    f"Balance violations (snapshots where no-comment files > commented files): {balance_violations}"
)
print()
dep_pct = 100 * num_files_with_deps / num_files if num_files else 0.0
avg_deps = total_dep_files / num_files_with_deps if num_files_with_deps else 0.0
print(f"Files with >=1 dependency:      {num_files_with_deps} ({dep_pct:.1f}%)")
print(f"Total dependency file entries:  {total_dep_files}")
print(f"Avg dependencies per file:      {avg_deps:.2f}")
print()
meta_pct = 100 * num_commits_with_metadata / num_snapshot_commits if num_snapshot_commits else 0.0
avg_meta = total_metadata_files / num_commits_with_metadata if num_commits_with_metadata else 0.0
print(f"Commits with metadata files:    {num_commits_with_metadata} ({meta_pct:.1f}%)")
print(f"Total metadata file entries:    {total_metadata_files}")
print(f"Avg metadata files per commit:  {avg_meta:.2f}")

Number of PRs:                  2
Number of snapshot commits:     2
Number of files (total):        4
  - with comments:              2
  - without comments:           2
Number of comments (total):     7
  - resolved:                   3
  - unresolved:                 4
Balance violations (snapshots where no-comment files > commented files): 0

Files with >=1 dependency:      4 (100.0%)
Total dependency file entries:  12
Avg dependencies per file:      3.00

Commits with metadata files:    2 (100.0%)
Total metadata file entries:    9
Avg metadata files per commit:  4.50


In [10]:
# Convert dataset to JSON with list of files
files_list = []

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        pr_title = pr_entry.get("pr_title")
        pr_body = pr_entry.get("pr_body")
        repo_star_count = pr_entry.get("repo_star_count")
        for commit_sha, path_map in pr_entry["commits"].items():
            # {path: content} map of well-known project metadata files (README,
            # requirements, pyproject.toml, etc.) found anywhere in the repo at
            # this snapshot commit.  Content is an annotated diff (base→head) when
            # the file changed between the PR base and this snapshot; otherwise raw
            # HEAD text.  Absent when no metadata files were found.
            commit_metadata_files = path_map.get(dataset_config.METADATA_FILES_COMMIT_KEY)
            for path, file_entry in path_map.items():
                if path == dataset_config.METADATA_FILES_COMMIT_KEY:
                    continue
                file_obj = {
                    "repo": repo_name,
                    "pr_number": pr_number,
                    "pr_title": pr_title,
                    "pr_body": pr_body,
                    "repo_star_count": repo_star_count,
                    "commit_sha": commit_sha,
                    "path": path,
                    "patched_content": file_entry.get("patched_content"),
                    # {path: content} map of direct in-repo imported files resolved
                    # from the HEAD version of this file; absent when none resolved.
                    "dependencies": file_entry.get("dependencies"),
                    # {path: content} map of well-known project metadata files found
                    # in the repo at this snapshot commit; shared across all files in
                    # the same commit.
                    "metadata_files": commit_metadata_files,
                    "comments": [
                        {
                            "body": comment.get("body"),
                            "is_resolved": comment.get("is_resolved"),
                            "annotated_start_line": comment.get("annotated_start_line"),
                            "annotated_end_line": comment.get("annotated_end_line"),
                        }
                        for comment in file_entry.get("comments", [])
                    ],
                }
                files_list.append(file_obj)
with open("files_list.json", "w") as f:
    json.dump(files_list, f)

In [ ]:
with open("./files_list.json", "r") as f:
    files = json.load(f)